# MindFlow Step 9a2 · Qwen volatility pilot

Test pretrained Qwen2.5-VL-3B-Instruct on two representations of the same v4 validation windows. The 6b-style input is a stock chart with an EWSD band plus 43 historical values in text. The 6d-style input is the full chart-and-strips canvas. The model scores whether the next five-minute absolute return is above the stock's training-period median.

Zero-shot reconstructs pilot windows and fits preprocessing on a small training subset. The frozen probe reuses the zero-shot validation keys and fitted scales. LoRA reuses the frozen probe's training and validation keys. A matched CNN comparison and held-out evaluation are follow-up stages.

## 1. Paths and configuration

Select a GPU kernel that can import PyTorch, Transformers with Qwen2.5-VL support, NumPy, pandas, SciPy, scikit-learn, PyArrow, Pillow and matplotlib. On NYU HPC use the project Python inside the Singularity container with GPU passthrough. The same runner can be called from a Terminal with `python MindFlow/code/qwen_eval.py --help`.

The defaults below use the v4 bundle already staged on NYU scratch. Set the paths explicitly if using another runtime.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/scratch') / os.environ['USER'] / 'image-trading'
MINDFLOW = PROJECT_ROOT / 'ImageTrading-EC-Project' / 'MindFlow'
PREPARED = PROJECT_ROOT / 'data/mindflow-v4/bundle/prepared'
BUNDLE = PROJECT_ROOT / 'data/alldata_bundle_v4.zip'
OUTPUT = MINDFLOW / 'runs/step9a_qwen'
N_ROWS = 32
SCALE_ROWS = 2000
SEED = 69
MODEL_REVISION = 'main'  # supply a commit SHA to pin a later rerun
MODE = 'zero_shot'  # or 'probe' or 'lora'
PROBE_REFERENCE = OUTPUT / 'qwen_zero_shot_20261128_025135_734169'
PROBE_TRAIN_ROWS = 2000
SHARED_TRAIN = None  # Path to train_keys.parquet for a shared validation reference
RUN_ID = None  # keep the same ID and settings when restarting probe or LoRA
LORA_REFERENCE = OUTPUT / 'qwen_probe_20261128_032030_316176'
LORA_STYLE = '6d'
LORA_EPOCHS = 3
LORA_MAX_TRAIN = None  # set to 32 for a smoke test

assert (MINDFLOW / 'code/qwen_eval.py').is_file(), MINDFLOW
assert (PREPARED / 'manifest.json').is_file(), PREPARED
assert BUNDLE.is_file(), BUNDLE
sys.path.insert(0, str(MINDFLOW / 'code'))

## 2. Score both inputs

The assistant's first-token logits for A (above median) and B are converted to a two-choice probability. Each run saves paired sample keys, labels, both logits and probabilities, AUC, log loss, Brier score, environment settings and parameter counts. Images are rendered in memory.

In [ ]:
if MODE == 'zero_shot':
    from qwen_eval import run
    run(PREPARED, BUNDLE, OUTPUT, n_rows=N_ROWS, scale_rows=SCALE_ROWS,
        seed=SEED, revision=MODEL_REVISION)
elif MODE == 'probe':
    from qwen_probe import run_probe
    run_probe(PREPARED, BUNDLE, OUTPUT, PROBE_REFERENCE, PROBE_TRAIN_ROWS,
              shared_train=SHARED_TRAIN, run_id=RUN_ID)
elif MODE == 'lora':
    from qwen_lora import run_lora  # requires peft
    run_lora(PREPARED, BUNDLE, OUTPUT, LORA_REFERENCE, LORA_STYLE, LORA_EPOCHS,
             LORA_MAX_TRAIN, run_id=RUN_ID)
else:
    raise ValueError(f'Unknown mode: {MODE}')